In [0]:
# define the catalog and database
catalog = 'uk_property_insights'
database = 'default'
bronze_table = 'bronze_property_prices'
silver_table = "silver_property_prices"

# set the catalog and database
spark.catalog.setCurrentCatalog(catalog)
spark.catalog.setCurrentDatabase(database)
print(f"Current catalog: {spark.catalog.currentCatalog()}")
print(f"Current database: {spark.catalog.currentDatabase()}")

# check if the tables exist
bronze_table_exists = spark.catalog.tableExists(bronze_table)
silver_table_exists = spark.catalog.tableExists(silver_table)

# exit the notebook if the tables do not exist
if not bronze_table_exists:
    raise Exception(f"Table {bronze_table} does not exist.")
if not silver_table_exists:
    raise Exception(f"Table {silver_table} does not exist.")

In [0]:
%sql
SELECT COUNT(*) FROM uk_property_insights.default.silver_property_prices;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Incremental processing
bronze_df = spark.table(bronze_table)
silver_df = spark.table(silver_table)

# Get latest processed timestamp
max_ts = silver_df.agg({"ingestion_time": "max"}).collect()[0][0]

if max_ts is None:
    # Silver table is empty, process all bronze records
    new_bronze_df = bronze_df
else:
    # Filter for records newer than the last processed timestamp
    new_bronze_df = bronze_df.filter(F.col("ingestion_time") > max_ts)

new_bronze_row_count = new_bronze_df.count()
print(f"New bronze records = {new_bronze_row_count}")


In [0]:
# Exit if no new bronze records
if new_bronze_row_count == 0:
    print("No new records to process")
    dbutils.notebook.exit("No new records to process")

# Define filter conditions for valid records
valid_condition = (
    (F.col("transaction_id").isNotNull()) &
    (F.col("price").isNotNull()) &
    (F.col("price") > 0) &
    (F.col("postcode").isNotNull()) &
    (F.col("transfer_date").isNotNull()) &
    (F.col("transfer_date") > F.lit("2020-01-01"))
)

# Clean and transform the new records from the bronze table
new_silver_df = (
    new_bronze_df
    .filter(valid_condition)
    .withColumn("row_num", F.row_number().over(
        Window.partitionBy("transaction_id").orderBy(F.col("ingestion_time").desc())
    ))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

# Capture filtered-out rows due to invalid/nulls
filtered_out_df = new_bronze_df.filter(~valid_condition)
display(filtered_out_df.limit(10))

# Apply trim and uppercase to string columns using withColumns
string_cols = [
    "primary_addressable_object_name", "secondary_addressable_object_name",
    "street", "locality", "town_city", "district", "county"
]
new_silver_df = new_silver_df.withColumns(
    {col: F.upper(F.trim(F.col(col))) for col in string_cols}
)

display(new_silver_df.limit(10))
new_silver_row_count = new_silver_df.count()
print(f"New silver records = {new_silver_row_count}")

In [0]:
# Merge (upsert) using Delta Lake
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, silver_table)

delta_table.alias("target").merge(
    new_silver_df.alias("source"),
    "target.transaction_id = source.transaction_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Merge operation completed successfully")

In [0]:
# Get the latest operation metrics from Delta history
history_df = delta_table.history(1)

# Extract operation metrics
latest_op = history_df.select("operation", "operationMetrics").collect()[0]
operation = latest_op["operation"]
metrics = latest_op["operationMetrics"]

print(f"Latest operation: {operation}")
print("Merge metrics:")
if metrics:
    print(f"  Rows inserted: {metrics.get('numTargetRowsInserted', 'N/A')}")
    print(f"  Rows updated: {metrics.get('numTargetRowsUpdated', 'N/A')}")
    print(f"  Rows deleted: {metrics.get('numTargetRowsDeleted', 'N/A')}")
    print(f"  Files added: {metrics.get('numTargetFilesAdded', 'N/A')}")
    print(f"  Files removed: {metrics.get('numTargetFilesRemoved', 'N/A')}")

In [0]:
%sql
SELECT COUNT(*) FROM uk_property_insights.default.silver_property_prices;